## 题目背景

在材料科学研究中，预测金属材料在复杂力学环境下的疲劳寿命是一个重要且具有实际价值的课题。疲劳寿命的预测能够帮助工程师在设计中合理选择材料，提高结构件的可靠性和安全性。本题基于一个高质量的实验数据集，旨在探索不同金属材料的特性与疲劳寿命之间的关系，并通过 **深度学习** 模型实现准确的寿命预测。



## **数据集概述**
题目所用数据收集自公开数据集，经过筛选和整理，涵盖 **40 种金属材料**。这些材料类型包括：铝, 钛, 镁, 铜, 镍合金、不锈钢和合金钢等等。

### **数据特征**
- **数据集**:`train.csv`中包含`822`行`487`列:
    - 前`482`列是每个材料的应力应变实验数据点；
    - `483`列 ~ `486`列分别表示材料的**弹性模量**、**抗拉强度**、**屈服强度**、**泊松比**；
    - 最后一列(`487`列)是要预测的标签，表示材料的疲劳寿命 $lg(N_f)$。

- **疲劳寿命 $N_f$**：
   - $N_f$ 表示材料在一定应力或应变条件下，从加载开始到发生疲劳断裂所能承受的循环次数。
   - $lg(N_f)$ 是 $N_f$ 的常用对数表示，将循环次数从指数级缩放为线性级数，方便分析。

## **题目要求**
- 请使用任意深度学习模型（PyTorch、TensorFlow等）基于训练集搭建深度学习模型，将模型保存为模型文件`model.pth`(格式支持`.pth`，`.h5`，`.onnx`，`.pkl`)；
- 使用深度学习模型对测试集数据进行预测，预测结果保存为`submission.csv`文件
- 最后，请将所有生成文件打包成`submission.zip`压缩文件

## 评分标准（总分30分）
1. **模型使用正常**：得 $5$ 分   

2. 均方根误差计算公式：  
**均方根误差**(Root Mean Square Error, RMSE)是评估回归模型预测值与真实值之间误差的一种常用指标，反映了预测值与实际值之间的偏差程度。

$$
\text{RMSE} = \sqrt{\frac{1}{n} \sum_{i=1}^{n} (y_i - \hat{y}_i)^2}
$$

<span style="padding-left: 3.5ch;">**参数解释**：$ n $：样本数量；$ y_i $：真实值（观测值）；$ \hat{y}_i $：预测值（模型输出） </span>


3. **RMSE评分**：
    - **RMSE** 低于 $0.6$: 给 $25$ 分
    - **RMSE** 为 $0.8$ ～ $0.6$（不包括$0.6$）：给 $20$ 分
    - **RMSE** 高于 $0.8$（不包括$0.8$）：给 $10$ 分

没有使用深度学习模型、没有结果或者显示错误：不得分

## 读取数据集

请自行设计数据集构建与读取方式，此处不做限制。

In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, RobustScaler
import os
import random
import warnings
warnings.filterwarnings('ignore')

# 设置随机种子保证结果可复现
def set_seed(seed=42):
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    np.random.seed(seed)
    random.seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# 构建数据集类
class FatigueDataset(Dataset):
    def __init__(self, curve_features, scalar_features, targets):
        # curve_features: (N, 482), 增加通道维度变为 (N, 1, 482) 供 1D-CNN 使用
        self.curve = torch.tensor(curve_features, dtype=torch.float32).unsqueeze(1)
        self.scalar = torch.tensor(scalar_features, dtype=torch.float32)
        self.targets = torch.tensor(targets, dtype=torch.float32).unsqueeze(1)

    def __len__(self):
        return len(self.targets)

    def __getitem__(self, idx):
        return self.curve[idx], self.scalar[idx], self.targets[idx]

In [3]:
# 加载训练数据
train_path = "/bohr/stress-yqqa/v4/train.csv"
df_train = pd.read_csv(train_path)

X = df_train.iloc[:, :-1].values  # 前486列
y = df_train.iloc[:, -1].values   # 第487列：疲劳寿命 lg(Nf)

# 分离曲线特征(前482列)和标量特征(后4列)
X_curve = X[:, :482]
X_scalar = X[:, 482:]

In [4]:
# 数据划分 (80% 训练, 20% 验证)
X_curve_train, X_curve_val, X_scalar_train, X_scalar_val, y_train, y_val = train_test_split(
    X_curve, X_scalar, y, test_size=0.2, random_state=42
)

# 数据标准化 (分离处理)
scaler_curve = RobustScaler()   # 对曲线噪声更鲁棒
scaler_scalar = StandardScaler() # 常规标准化

X_curve_train_scaled = scaler_curve.fit_transform(X_curve_train)
X_curve_val_scaled = scaler_curve.transform(X_curve_val)

X_scalar_train_scaled = scaler_scalar.fit_transform(X_scalar_train)
X_scalar_val_scaled = scaler_scalar.transform(X_scalar_val)

# 构建 DataLoader (num_workers设为0避免某些Jupyter环境报错)
train_dataset = FatigueDataset(X_curve_train_scaled, X_scalar_train_scaled, y_train)
val_dataset = FatigueDataset(X_curve_val_scaled, X_scalar_val_scaled, y_val)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=0)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=0)# 数据划分 (80% 训练, 20% 验证)
X_curve_train, X_curve_val, X_scalar_train, X_scalar_val, y_train, y_val = train_test_split(
    X_curve, X_scalar, y, test_size=0.2, random_state=42
)

# 数据标准化 (分离处理)
scaler_curve = RobustScaler()   # 对曲线噪声更鲁棒
scaler_scalar = StandardScaler() # 常规标准化

X_curve_train_scaled = scaler_curve.fit_transform(X_curve_train)
X_curve_val_scaled = scaler_curve.transform(X_curve_val)

X_scalar_train_scaled = scaler_scalar.fit_transform(X_scalar_train)
X_scalar_val_scaled = scaler_scalar.transform(X_scalar_val)

# 构建 DataLoader (num_workers设为0避免某些Jupyter环境报错)
train_dataset = FatigueDataset(X_curve_train_scaled, X_scalar_train_scaled, y_train)
val_dataset = FatigueDataset(X_curve_val_scaled, X_scalar_val_scaled, y_val)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=0)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=0)

## 搭建模型并训练
这里请使用 **深度学习** 模型，对训练数据进行模型训练，**将性能最佳的模型保存为<font color="red">`model.pth`</font>**(格式支持`.pth`，`.h5`，`.onnx`，`.pkl`)，推荐使用 **PyTorch**。

In [5]:
class ResBlock1D(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size=5):
        super().__init__()
        self.conv1 = nn.Conv1d(in_channels, out_channels, kernel_size, padding=kernel_size//2)
        self.bn1 = nn.BatchNorm1d(out_channels)
        self.act1 = nn.GELU()
        self.conv2 = nn.Conv1d(out_channels, out_channels, kernel_size, padding=kernel_size//2)
        self.bn2 = nn.BatchNorm1d(out_channels)
        self.act2 = nn.GELU()
        self.downsample = nn.Conv1d(in_channels, out_channels, 1) if in_channels != out_channels else nn.Identity()
        
    def forward(self, x):
        identity = self.downsample(x)
        out = self.act1(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        out += identity
        return self.act2(out)

class FatiguePredictor(nn.Module):
    def __init__(self):
        super(FatiguePredictor, self).__init__()
        # 1. 1D CNN 提取应力应变曲线特征
        self.stem = nn.Sequential(
            nn.Conv1d(1, 32, kernel_size=7, padding=3),
            nn.BatchNorm1d(32), nn.GELU(), nn.MaxPool1d(2)
        )
        self.layer1 = ResBlock1D(32, 64)
        self.pool1 = nn.MaxPool1d(2)
        self.layer2 = ResBlock1D(64, 128)
        self.pool2 = nn.MaxPool1d(2)
        self.layer3 = ResBlock1D(128, 256)
        self.pool3 = nn.AdaptiveAvgPool1d(1)
        
        # 2. MLP 提取材料力学参数特征
        self.mlp_scalar = nn.Sequential(
            nn.Linear(4, 16), nn.GELU(),
            nn.Linear(16, 32), nn.GELU()
        )
        
        # 3. 特征融合与回归头
        self.regressor = nn.Sequential(
            nn.Linear(256 + 32, 128),
            nn.BatchNorm1d(128), nn.GELU(), nn.Dropout(0.3),
            nn.Linear(128, 64), nn.GELU(), nn.Dropout(0.2),
            nn.Linear(64, 1)
        )

    def forward(self, curve, scalar):
        x = self.stem(curve)
        x = self.pool1(self.layer1(x))
        x = self.pool2(self.layer2(x))
        x = self.pool3(self.layer3(x)).squeeze(-1)
        
        s = self.mlp_scalar(scalar)
        combined = torch.cat([x, s], dim=1)
        return self.regressor(combined)

model = FatiguePredictor().to(device)

In [6]:
# 使用 HuberLoss 训练，对异常值更鲁棒，有助于降低整体 RMSE
criterion = nn.HuberLoss(delta=1.0) 
optimizer = optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=50, eta_min=1e-5)

best_val_rmse = float('inf')
patience = 30
trigger_times = 0
epochs = 150 # 增加 Epoch 确保充分收敛

for epoch in range(epochs):
    model.train()
    train_loss = 0.0
    for curve_batch, scalar_batch, y_batch in train_loader:
        curve_batch, scalar_batch, y_batch = curve_batch.to(device), scalar_batch.to(device), y_batch.to(device)
        optimizer.zero_grad()
        outputs = model(curve_batch, scalar_batch)
        loss = criterion(outputs, y_batch)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()

    model.eval()
    val_loss = 0.0
    preds, targets = [], []
    with torch.no_grad():
        for curve_batch, scalar_batch, y_batch in val_loader:
            curve_batch, scalar_batch, y_batch = curve_batch.to(device), scalar_batch.to(device), y_batch.to(device)
            outputs = model(curve_batch, scalar_batch)
            val_loss += criterion(outputs, y_batch).item()
            preds.append(outputs.cpu().numpy())
            targets.append(y_batch.cpu().numpy())

    avg_train_loss = train_loss / len(train_loader)
    preds = np.concatenate(preds)
    targets = np.concatenate(targets)
    val_rmse = np.sqrt(np.mean((preds - targets)**2)) # 计算真实的 RMSE 指标

    scheduler.step()
    print(f"Epoch [{epoch+1}/{epochs}] | Train Loss: {avg_train_loss:.4f} | Val RMSE: {val_rmse:.4f} | LR: {optimizer.param_groups[0]['lr']:.6f}")

    if val_rmse < best_val_rmse:
        best_val_rmse = val_rmse
        trigger_times = 0
        torch.save(model.state_dict(), 'model.pth')
        print(f"✅ Best model saved! Val RMSE: {val_rmse:.4f}")
    else:
        trigger_times += 1
        if trigger_times >= patience:
            print(f"⏹️ Early stopping at epoch {epoch+1}")
            break

print(f"Training finished. Best Val RMSE: {best_val_rmse:.4f}")

In [48]:
# 请务必将性能最佳的模型保存为"model.pth"，推荐使用PyTorch
# torch.save(model.state_dict(), 'model.pth')

## 测试数据预测
使用模型对只含有 **特征** 而不含 **标签** 的测试数据`test_no_label.csv`进行预测，并将预测结果保存为`submission.csv`，后台中会把你代码预测的结果与测试集的真实数据（`test.csv`）进行比对并评分。

选手可以参考下方代码对不含标签的测试数据(`test_no_label.csv`)进行预测，**该数据的路径已经提供**，**选手在调试代码时可以使用路径`/bohr/stress-yqqa/v2/test_no_label.csv`访问**。           
注意，`DATA_PATH`数据路径的部分请不要修改，避免评分运行报错。

In [12]:
import os
#-----读取训练集，训练集的地址已设置，以下部分无需修改-------#
train_path = "/bohr/stress-yqqa/v4/train.csv" 
data_train = pd.read_csv(train_path)  

# 读取无标签测试集（也就是测试数据的特征），请确保路径使用正确，避免测试报错
test_no_label_path = "/bohr/stress-yqqa/v4/test_no_label.csv"
data_test = pd.read_csv(test_no_label_path) 


In [14]:
# 注意，如果对训练数据进行了标准化，
# 请对测试数据同样进行标准化处理并转换成张量形式（PyTorch）

# 分离并标准化测试集特征
X_test = data_test.values
X_test_curve = X_test[:, :482]
X_test_scalar = X_test[:, 482:]

X_test_curve_scaled = scaler_curve.transform(X_test_curve)
X_test_scalar_scaled = scaler_scalar.transform(X_test_scalar)

test_curve_tensor = torch.tensor(X_test_curve_scaled, dtype=torch.float32).unsqueeze(1).to(device)
test_scalar_tensor = torch.tensor(X_test_scalar_scaled, dtype=torch.float32).to(device)

# 加载最佳模型
model.load_state_dict(torch.load('model.pth', map_location=device, weights_only=True))
model.eval()



In [15]:
# 预测测试集（代码仅供参考，可以依据你的代码修改）
predictions = []
with torch.no_grad():
    batch_size = 32
    for i in range(0, len(test_curve_tensor), batch_size):
        curve_batch = test_curve_tensor[i:i+batch_size]
        scalar_batch = test_scalar_tensor[i:i+batch_size]
        pred = model(curve_batch, scalar_batch).cpu().numpy().flatten()
        predictions.extend(pred)

In [ ]:
# 保存预测结果（预测结果的列名必须为 "Predicted", 文件名必须为 "submission.csv"）
data_test['Predicted'] = predictions
data_test.to_csv('submission.csv', index=False)

最后，需要将刚刚输出的 **<font color="red">`model.pth`</font>** 和 **<font color = "red">`submission.csv`</font>**，打包成一个zip压缩包，文件名为 **<font color="red">`submission.zip`</font>**.

In [ ]:
import zipfile
import os

# 定义要打包的文件和压缩文件名 
files_to_zip = ['submission.csv', 'model.pth']
zip_filename = 'submission.zip'

# 创建一个 zip 文件
with zipfile.ZipFile(zip_filename, 'w') as zipf:
    for file in files_to_zip:
        # 将文件添加到 zip 文件中, Add files to the zip file
        zipf.write(file, os.path.basename(file))

print(f'{zip_filename} is created successfully!')